# 🏓 TTNet – Player Performance Analysis
**نظام تحليل أداء لاعب تنس الطاولة**

---
### خطوات هذا الـ Notebook
1. ✅ ربط Google Drive
2. ✅ رفع الكود
3. ✅ تثبيت المتطلبات
4. ✅ تحميل الداتا
5. ✅ استخراج الفريمات
6. ✅ التدريب (3 مراحل)
7. ✅ تشغيل تحليل أداء اللاعب

> **ملاحظة:** تأكدي من تفعيل GPU قبل البدء:
> `Runtime → Change runtime type → T4 GPU`

## 0️⃣ تحقق من الـ GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️  لا يوجد GPU! تأكدي من Runtime → Change runtime type → T4 GPU')

## ⚙️ إعدادات المشروع — شغّلي هذه الخلية أولاً دائماً
> إذا انقطعت الجلسة أو أعدتِ تشغيل الـ Kernel، شغّلي هذه الخلية مباشرةً قبل أي خلية أخرى.

In [ ]:
import os, subprocess
from pathlib import Path

# ══════════════════════════════════════════════════════════════════
#   ← غيري القيم هنا فقط
# ══════════════════════════════════════════════════════════════════

PROJECT_DIR  = '/content/drive/MyDrive/TTNet'
TRAIN_GAME   = 'game_1'    # game_1 إلى game_5
TEST_GAME    = 'test_2'    # test_2 = أصغر حجم (0.2 GB)

# رابط GitHub — ضعي رابط الـ repo تبعك هنا
# مثال: 'https://github.com/ayaalghadban/TTNet.git'
GITHUB_URL   = ''          # ← اتركيه فارغ لو بدك ترفعي ZIP

# ══════════════════════════════════════════════════════════════════
#   متغيرات مشتقة — لا تعدّليها
# ══════════════════════════════════════════════════════════════════

DATASET_DIR     = f'{PROJECT_DIR}/dataset'
CODE_DIR        = f'{PROJECT_DIR}/code'
CHECKPOINTS_DIR = f'{PROJECT_DIR}/checkpoints'
LOGS_DIR        = f'{PROJECT_DIR}/logs'
RESULTS_DIR     = f'{PROJECT_DIR}/results'
BASE_URL        = 'https://lab.osai.ai/datasets/openttgames/data/'

for d in [PROJECT_DIR, DATASET_DIR, CHECKPOINTS_DIR, LOGS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── جلب الكود ─────────────────────────────────────────────────────
def _find_src(base):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files and 'config' in dirs:
            return root
    return None

SRC_DIR = _find_src(CODE_DIR) if os.path.isdir(CODE_DIR) else None

if SRC_DIR:
    print(f'✅ الكود موجود مسبقاً: {SRC_DIR}')

elif GITHUB_URL:
    # ── خيار أ: GitHub ────────────────────────────────────────────
    print(f'📥 استنساخ الكود من GitHub...')
    subprocess.run(['git', 'clone', GITHUB_URL, CODE_DIR], check=True)
    SRC_DIR = _find_src(CODE_DIR)
    print(f'✅ تم استنساخ الكود: {SRC_DIR}')

else:
    # ── خيار ب: ZIP من Drive ──────────────────────────────────────
    ZIP_PATH = f'{PROJECT_DIR}/TTNet.zip'
    if os.path.isfile(ZIP_PATH):
        print('📦 فك ضغط الكود من ZIP...')
        subprocess.run(['unzip', '-q', ZIP_PATH, '-d', CODE_DIR], check=True)
        SRC_DIR = _find_src(CODE_DIR)
        print(f'✅ تم فك الضغط: {SRC_DIR}')
    else:
        print('⚠️  الكود غير موجود. عندك خيارين:')
        print('   1) ضعي رابط GitHub في GITHUB_URL أعلاه')
        print(f'   2) ارفعي ZIP على: {ZIP_PATH}')

ROOT_DIR = str(Path(SRC_DIR).parent) if SRC_DIR else CODE_DIR

# ── ملخص ──────────────────────────────────────────────────────────
print('\n📋 الإعدادات الحالية:')
print(f'   PROJECT_DIR  = {PROJECT_DIR}')
print(f'   TRAIN_GAME   = {TRAIN_GAME}  |  TEST_GAME = {TEST_GAME}')
print(f'   SRC_DIR      = {SRC_DIR or "❌ غير موجود"}')
print(f'   DATASET_DIR  = {DATASET_DIR}')

## 1️⃣ ربط Google Drive
كل الملفات (كود، داتا، نماذج) ستُحفظ على Drive بحيث لا تضيع عند إغلاق الجلسة.

In [ ]:
from google.colab import drive
import os

if os.path.isdir('/content/drive/MyDrive'):
    print('✅ Drive مربوط مسبقاً.')
else:
    drive.mount('/content/drive')
    print('✅ تم ربط Drive.')

print('   مجلد المشروع:', PROJECT_DIR)

## 2️⃣ رفع الكود

**عندك خيارين:**

### الخيار أ – رفع ملف ZIP (الأسرع)
1. اضغطي على مجلد المشروع على جهازك
2. اعمليه ZIP
3. ارفعيه على: `MyDrive/TTNet/TTNet.zip`
4. شغّلي الخلية أدناه

### الخيار ب – GitHub (لو رفعتي الكود)
```python
!git clone https://github.com/YOUR_USERNAME/TTNet.git {PROJECT_DIR}/code
```

In [ ]:
# ── الخيار أ: فك ضغط الـ ZIP ──────────────────────────────────────────────────
ZIP_PATH  = f'{PROJECT_DIR}/TTNet.zip'
CODE_DIR  = f'{PROJECT_DIR}/code'

if os.path.isfile(ZIP_PATH):
    print('📦 فك ضغط الكود...')
    !unzip -q "{ZIP_PATH}" -d "{CODE_DIR}"
    print('✅ تم فك الضغط في:', CODE_DIR)
elif os.path.isdir(CODE_DIR):
    print('✅ الكود موجود مسبقاً في:', CODE_DIR)
else:
    print('⚠️  لم يُعثر على ZIP. ارفعي الملف على Drive أو استخدمي GitHub.')

# ── الخيار ب: GitHub ───────────────────────────────────────────────────────────
# GITHUB_URL = 'https://github.com/YOUR_USERNAME/TTNet.git'
# !git clone {GITHUB_URL} {CODE_DIR}

## 3️⃣ تثبيت المتطلبات

In [ ]:
print('📦 تثبيت المكتبات المطلوبة...')

# المكتبات الأساسية (Colab لديه torch مثبت مسبقاً)
!pip install -q easydict==1.9 scikit-learn tqdm

# TurboJPEG – نثبّت الإصدار 1.x المتوافق مع libjpeg-turbo 2.x الموجود في Colab
!apt-get install -q -y libturbojpeg
!pip install -q "PyTurboJPEG==1.7.1"

# tensorboard
!pip install -q tensorboard

# تحقق من الإصدارات
import subprocess
result = subprocess.run(['python3', '-c',
    'from turbojpeg import TurboJPEG; tj = TurboJPEG(); print("✅ TurboJPEG يعمل بشكل صحيح")'],
    capture_output=True, text=True)
print(result.stdout or result.stderr)
print('✅ جميع المكتبات مثبتة')

## 4️⃣ إصلاح مشاكل التوافق مع NumPy الحديث
الكود الأصلي يستخدم `np.int` المحذوف من NumPy 1.24+. نصلح هذا تلقائياً.

In [ ]:
import subprocess, os, re

# اجد المسار الصحيح للكود
# الكود قد يكون في مجلد فرعي داخل code/ حسب اسم ZIP
def find_src_dir(base):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files and 'config' in dirs:
            return root
    return None

SRC_DIR = find_src_dir(CODE_DIR)
if SRC_DIR is None:
    # fallback: try common patterns
    for sub in os.listdir(CODE_DIR):
        candidate = os.path.join(CODE_DIR, sub, 'src')
        if os.path.isdir(candidate):
            SRC_DIR = candidate
            break

print('📁 SRC_DIR:', SRC_DIR)
ROOT_DIR = os.path.dirname(SRC_DIR) if SRC_DIR else CODE_DIR

# إصلاح np.int → int في جميع ملفات Python
import re
from pathlib import Path

fixed = []
for py_file in Path(CODE_DIR).rglob('*.py'):
    text = py_file.read_text(errors='ignore')
    new_text = re.sub(r'dtype=np\.int\b', 'dtype=np.int32', text)
    new_text = re.sub(r'\.astype\(np\.int\)', '.astype(np.int32)', new_text)
    new_text = re.sub(r'np\.int\b(?!\d|e|_)', 'int', new_text)
    if new_text != text:
        py_file.write_text(new_text)
        fixed.append(str(py_file))

if fixed:
    print(f'✅ تم إصلاح {len(fixed)} ملفات:')
    for f in fixed:
        print('   ', f)
else:
    print('✅ لا توجد مشاكل توافق')

# ── إصلاح 2: تخطي الأحداث التي تنقصها صور (يتيح التدريب بأي عدد صور) ─────────
_data_utils = Path(SRC_DIR) / 'data_process' / 'ttnet_data_utils.py'
_du_text = _data_utils.read_text()
_skip_check = "                # Skip sequences where any of the 9 image files is missing\n                if not all(os.path.isfile(p) for p in img_path_list):\n                    continue\n\n"
if _skip_check not in _du_text:
    _old = "                # Get segmentation path for the last frame in the sequence"
    _du_text = _du_text.replace(_old, _skip_check + _old)
    _data_utils.write_text(_du_text)
    print('✅ تم إضافة فلتر الصور المفقودة في ttnet_data_utils.py')
else:
    print('✅ فلتر الصور المفقودة موجود مسبقاً')

## 5️⃣ تحميل الداتا (OpenTTGames) – فيديو واحد فقط

المصدر: [lab.osai.ai](https://lab.osai.ai/)

اختاري **فيديو تدريب واحد** و**فيديو اختبار واحد** من الجدول التالي:

| الاسم | الحجم | النوع |
|-------|-------|-------|
| `game_1` | 5.6 GB | تدريب |
| `game_2` | 10.8 GB | تدريب |
| `game_3` | 4.6 GB | تدريب |
| `game_4` | 3.9 GB | تدريب |
| `game_5` | 4.5 GB | تدريب |
| `test_2` | **0.2 GB** ✅ أصغر اختبار | اختبار |
| `test_3` | 0.5 GB | اختبار |
| `test_1` | 1.1 GB | اختبار |

> **نصيحة:** استخدمي `game_1` للتدريب و`test_2` للاختبار – هما الأنسب للكولاب المجاني.

In [ ]:
# المجلدات
for d in ['training/videos', 'training/annotations',
          'test/videos',     'test/annotations']:
    os.makedirs(f'{DATASET_DIR}/{d}', exist_ok=True)

def download_if_missing(url, dst, label):
    if os.path.isfile(dst):
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  [skip] {label} موجود مسبقاً ({size_mb:.0f} MB)')
        return
    print(f'  ⬇️  تحميل {label} ...')
    import subprocess
    subprocess.run(['wget', '-q', '--show-progress', url, '-O', dst], check=True)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'  ✅ تم تحميل {label} ({size_mb:.0f} MB)')

# ── فيديو التدريب ─────────────────────────────────────────────────────────────
print(f'📥 فيديو التدريب: {TRAIN_GAME}')
download_if_missing(
    url = f'{BASE_URL}{TRAIN_GAME}.mp4',
    dst = f'{DATASET_DIR}/training/videos/{TRAIN_GAME}.mp4',
    label = f'{TRAIN_GAME}.mp4'
)
download_if_missing(
    url = f'{BASE_URL}{TRAIN_GAME}.zip',
    dst = f'{DATASET_DIR}/training/annotations/{TRAIN_GAME}.zip',
    label = f'{TRAIN_GAME}.zip (annotations)'
)

# ── فيديو الاختبار ────────────────────────────────────────────────────────────
print(f'📥 فيديو الاختبار: {TEST_GAME}')
download_if_missing(
    url = f'{BASE_URL}{TEST_GAME}.mp4',
    dst = f'{DATASET_DIR}/test/videos/{TEST_GAME}.mp4',
    label = f'{TEST_GAME}.mp4'
)
download_if_missing(
    url = f'{BASE_URL}{TEST_GAME}.zip',
    dst = f'{DATASET_DIR}/test/annotations/{TEST_GAME}.zip',
    label = f'{TEST_GAME}.zip (annotations)'
)

print(f'\n✅ اكتمل التحميل')
print(f'   تدريب: {TRAIN_GAME} | اختبار: {TEST_GAME}')

## 6️⃣ فك ضغط الـ Annotations

In [ ]:
import zipfile, shutil

def unzip_annotations(annos_dir):
    """
    فك ضغط كل ZIP في annos_dir داخل مجلد فرعي باسم اللعبة.
    game_1.zip → annotations/game_1/ball_markup.json ...
    """
    for zip_fn in os.listdir(annos_dir):
        if not zip_fn.endswith('.zip'):
            continue

        game_name  = zip_fn[:-4]                               # game_1
        zip_path   = os.path.join(annos_dir, zip_fn)
        out_folder = os.path.join(annos_dir, game_name)        # annotations/game_1/

        if os.path.isdir(out_folder) and os.listdir(out_folder):
            print(f'  [skip] {game_name}/ موجود مسبقاً')
            continue

        os.makedirs(out_folder, exist_ok=True)
        print(f'  📂 فك {zip_fn} → {game_name}/ ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_folder)

        # لو الملفات اتفكت مباشرةً في annos_dir بشكل خاطئ، ننقلها
        for item in ['ball_markup.json', 'events_markup.json', 'segmentation_masks']:
            src = os.path.join(annos_dir, item)
            dst = os.path.join(out_folder, item)
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.move(src, dst)
                print(f'     نقل {item} → {game_name}/')

        # تحقق
        files = os.listdir(out_folder)
        print(f'  ✅ {game_name}/ يحتوي: {files}')

print('📂 فك ضغط annotations التدريب...')
unzip_annotations(f'{DATASET_DIR}/training/annotations')
print('📂 فك ضغط annotations الاختبار...')
unzip_annotations(f'{DATASET_DIR}/test/annotations')
print('✅ تم فك الضغط بالهيكل الصحيح')

## 6️⃣ب ضبط الـ Config على الفيديو الواحد المحمّل
الكود الأصلي يتوقع 5 فيديوهات تدريب + 7 اختبار. هنا نضبطه تلقائياً على الفيديو اللي حملناه.

In [ ]:
from pathlib import Path
import re

# مسار config.py
config_path = Path(SRC_DIR) / 'config' / 'config.py'
config_text = config_path.read_text()

# استبدال قوائم الألعاب المُشفّرة بالفيديو الواحد المحمّل
config_text = re.sub(
    r"configs\.train_game_list\s*=\s*\[.*?\]",
    f"configs.train_game_list = ['{TRAIN_GAME}']",
    config_text
)
config_text = re.sub(
    r"configs\.test_game_list\s*=\s*\[.*?\]",
    f"configs.test_game_list = ['{TEST_GAME}']",
    config_text
)

config_path.write_text(config_text)

# تحقق
for line in config_text.splitlines():
    if 'game_list' in line:
        print(' ', line.strip())

print('\n✅ config.py تم ضبطه على:')
print(f'   train_game_list = [{TRAIN_GAME}]')
print(f'   test_game_list  = [{TEST_GAME}]')

## 7️⃣ استخراج الفريمات
نستخرج فقط الفريمات المطلوبة بحد أقصى **128 فريم** لكل فيديو مع التركيز على **اللاعب الأيسر**.

In [ ]:
import sys, glob as _glob, shutil, re
PREPARE_DIR  = os.path.join(ROOT_DIR, 'prepare_dataset')
LOCAL_DATASET = '/content/dataset_local'
sys.path.insert(0, PREPARE_DIR)

# ── إصلاح default=128 في extract_selected_images.py ──────────────────────────
_extract_path = os.path.join(PREPARE_DIR, 'extract_selected_images.py')
_txt = open(_extract_path).read()
_txt_fixed = re.sub(r"default=128,\s*\n(\s*help=.*frames.*)", 
                     "default=99999,\n\\1", _txt)
if _txt_fixed != _txt:
    open(_extract_path, 'w').write(_txt_fixed)
    print('✅ تم إصلاح default max_frames → 99999')
else:
    print('✅ max_frames default لا يحتاج إصلاح')

# ── خطوة 1: نسخ الفيديو والـ annotations للـ local ───────────────────────────
for split, game in [('training', TRAIN_GAME), ('test', TEST_GAME)]:
    dst_vid = f'{LOCAL_DATASET}/{split}/videos/{game}.mp4'
    dst_ann = f'{LOCAL_DATASET}/{split}/annotations/{game}'
    os.makedirs(os.path.dirname(dst_vid), exist_ok=True)
    os.makedirs(f'{LOCAL_DATASET}/{split}/images', exist_ok=True)

    if not os.path.isfile(dst_vid):
        print(f'📋 نسخ {game}.mp4 للـ local (2-5 دقائق) ...')
        shutil.copy2(f'{DATASET_DIR}/{split}/videos/{game}.mp4', dst_vid)
        print(f'   ✅ تم')
    else:
        print(f'   [ok] {game}.mp4 موجود local')

    if not os.path.isdir(dst_ann):
        shutil.copytree(f'{DATASET_DIR}/{split}/annotations/{game}', dst_ann)
        print(f'   ✅ annotations/{game} نُسخت')
    else:
        print(f'   [ok] annotations/{game} موجودة local')

# ── خطوة 2: حذف الصور القديمة الناقصة (128 فقط) ─────────────────────────────
print('\n🗑️  حذف الصور القديمة الناقصة...')
for split, game in [('training', TRAIN_GAME), ('test', TEST_GAME)]:
    old_local = f'{LOCAL_DATASET}/{split}/images/{game}'
    old_drive = f'{DATASET_DIR}/{split}/images/{game}'
    for folder in [old_local, old_drive]:
        if os.path.isdir(folder):
            count = len(os.listdir(folder))
            if count < 500:   # ناقص → احذف وأعد
                shutil.rmtree(folder)
                print(f'   🗑️  حُذف {folder} ({count} صورة ناقصة)')
            else:
                print(f'   [ok] {folder} ({count} صورة كافية)')

# ── خطوة 3: استخراج كل الفريمات (~5000 لـ game_1) ───────────────────────────
%cd "{PREPARE_DIR}"

# ══════════════════════════════════════════════════════════
# عدد الفريمات لكل فيديو:
#   1000 → ~110 حدث قابل للتدريب، سريع (~5 دقائق)
#   2000 → ~220 حدث، متوسط (~10 دقائق)
#   5000 → ~550 حدث، كامل  (~20 دقيقة) ← للنتائج الأفضل
MAX_EXTRACT = 1000   # ← غيري هنا حسب وقتك
# ══════════════════════════════════════════════════════════

print(f'\n🖼️  استخراج {MAX_EXTRACT} فريم من {TRAIN_GAME} ...')
!python extract_selected_images.py \
    --dataset_dir "{LOCAL_DATASET}" \
    --dataset_types training \
    --max_frames {MAX_EXTRACT} \
    --num_frames_sequence 9

print(f'\n🖼️  استخراج فريمات {TEST_GAME} ...')
!python extract_selected_images.py \
    --dataset_dir "{LOCAL_DATASET}" \
    --dataset_types test \
    --max_frames 500 \
    --num_frames_sequence 9

# ── خطوة 4: نسخ الصور إلى Drive للحفظ الدائم ────────────────────────────────
print('\n📤 نسخ الصور إلى Drive...')
for split, game in [('training', TRAIN_GAME), ('test', TEST_GAME)]:
    src = f'{LOCAL_DATASET}/{split}/images/{game}'
    dst = f'{DATASET_DIR}/{split}/images/{game}'
    if os.path.isdir(src) and os.listdir(src):
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'   ✅ {game}: {len(os.listdir(dst))} صورة → Drive')
    else:
        print(f'   ⚠️  {game}: فارغ!')

# ── تحقق نهائي ────────────────────────────────────────────────────────────────
train_n = len(_glob.glob(f'{LOCAL_DATASET}/training/images/{TRAIN_GAME}/*.jpg'))
test_n  = len(_glob.glob(f'{LOCAL_DATASET}/test/images/{TEST_GAME}/*.jpg'))
print(f'\n{"✅" if train_n >= MAX_EXTRACT else "❌"} {TRAIN_GAME}: {train_n} صورة')
print(f'{"✅" if test_n  >= 200        else "❌"} {TEST_GAME}:  {test_n} صورة')

## 8️⃣ إعداد مجلدات الإخراج

## ✅ تحقق من الصور قبل التدريب — شغّلي هذه أولاً
**إذا طلع خطأ هنا لا تشغّلي التدريب.**

In [ ]:
import glob as _glob

LOCAL_DATASET = '/content/dataset_local'

train_local = _glob.glob(f'{LOCAL_DATASET}/training/images/{TRAIN_GAME}/*.jpg')
test_local  = _glob.glob(f'{LOCAL_DATASET}/test/images/{TEST_GAME}/*.jpg')
train_drive = _glob.glob(f'{DATASET_DIR}/training/images/{TRAIN_GAME}/*.jpg')

print('📊 عدد الصور المتاحة:')
print(f'   Local - {TRAIN_GAME} تدريب : {len(train_local)} صورة')
print(f'   Local - {TEST_GAME}  اختبار: {len(test_local)} صورة')
print(f'   Drive - {TRAIN_GAME} تدريب : {len(train_drive)} صورة')

# تحديد مصدر الداتا للتدريب
MIN_REQUIRED = 200   # الحد الأدنى للبدء بالتدريب

if len(train_local) >= MIN_REQUIRED:
    TRAIN_DATA_DIR = LOCAL_DATASET
    print(f'\n✅ سيتم التدريب من LOCAL: {LOCAL_DATASET}')
    print(f'   عدد الصور المتاحة للتدريب: {len(train_local)}')
elif len(train_drive) >= MIN_REQUIRED:
    TRAIN_DATA_DIR = DATASET_DIR
    print(f'\n⚠️  سيتم التدريب من Drive: {DATASET_DIR}')
else:
    TRAIN_DATA_DIR = None
    print(f'\n❌ الصور غير كافية (موجود: {len(train_local)}, مطلوب: {MIN_REQUIRED}+)')
    print('   شغّلي خلية 7️⃣ أولاً')
    raise SystemExit('⛔ شغّلي خلية 7 أولاً')

In [ ]:
print('📁 المجلدات:')
print('   Checkpoints:', CHECKPOINTS_DIR)
print('   Logs:       ', LOGS_DIR)
print('   Results:    ', RESULTS_DIR)

## 9️⃣ التدريب – المرحلة الأولى
**Global Ball Detection + Segmentation فقط**

- `--no_local` `--no_event` → لا تدريب على Local أو Events
- `--global_weight 2` → تركيز على الكرة العالمية
- `--num_epochs 30`

In [ ]:
%cd "{SRC_DIR}"

# TRAIN_DATA_DIR تُحدد تلقائياً من خلية التحقق (local أسرع، Drive احتياطي)
!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase1 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-3 \
    --lr_type plateau \
    --no_local \
    --no_event \
    --global_weight 2 \
    --seg_weight 1 \
    --smooth-labelling \
    --earlystop_patience 5 \
    --checkpoint_freq 2 \
    --num_workers 2

print('✅ انتهى تدريب المرحلة الأولى')

## 🔟 التدريب – المرحلة الثانية
**Local Ball Detection + Event Spotting**

- نحمّل نتائج المرحلة الأولى (`pretrained_path`)
- `--overwrite_global_2_local` → ننسخ أوزان Global لـ Local
- نجمّد Global + Segmentation

In [ ]:
PHASE1_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase1/ttnet_phase1_best.pth'

if not os.path.isfile(PHASE1_CKPT):
    # ابحثي عن أحدث checkpoint
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase1'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE1_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Checkpoint المرحلة الأولى:', PHASE1_CKPT)

%cd "{SRC_DIR}"

!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase2 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-3 \
    --lr_type plateau \
    --pretrained_path "{PHASE1_CKPT}" \
    --overwrite_global_2_local \
    --freeze_global \
    --freeze_seg \
    --local_weight 1 \
    --event_weight 1 \
    --smooth-labelling \
    --earlystop_patience 5 \
    --num_workers 2

print('✅ انتهى تدريب المرحلة الثانية')

## 1️⃣1️⃣ التدريب – المرحلة الثالثة (Fine-tuning الكل)
نفتح جميع الأوزان للتدريب معاً بـ learning rate منخفض.

In [ ]:
PHASE2_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase2/ttnet_phase2_best.pth'

if not os.path.isfile(PHASE2_CKPT):
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase2'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE2_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Checkpoint المرحلة الثانية:', PHASE2_CKPT)

%cd "{SRC_DIR}"

!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase3 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-4 \
    --lr_type plateau \
    --pretrained_path "{PHASE2_CKPT}" \
    --global_weight 1 \
    --local_weight 1 \
    --event_weight 1 \
    --seg_weight 1 \
    --smooth-labelling \
    --earlystop_patience 7 \
    --num_workers 2

print('✅ انتهى التدريب الكامل')

## 1️⃣2️⃣ مراقبة التدريب بـ TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{LOGS_DIR}"

## 1️⃣3️⃣ اختبار الموديل (Test)

In [ ]:
PHASE3_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase3/ttnet_phase3_best.pth'

if not os.path.isfile(PHASE3_CKPT):
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase3'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE3_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Best Checkpoint:', PHASE3_CKPT)

%cd "{SRC_DIR}"

!python test.py \
    --working-dir "{PROJECT_DIR}" \
    --gpu_idx 0 \
    --pretrained_path "{PHASE3_CKPT}" \
    --save_test_output

## 1️⃣4️⃣ تشغيل تحليل أداء اللاعب

**رفعي فيديو اللاعب على Drive** ثم حددي المسار أدناه.

- `--assessment_video` : فيديو التقييم الأولي (10 كرات)
- `--stage_videos`     : فيديوهات المراحل
- `--starting_level`  : مستوى اللاعب (beginner/intermediate/advanced)

In [ ]:
# ── حدّدي مسارات الفيديوهات ───────────────────────────────────────────────────
PLAYER_VIDEO = f'{PROJECT_DIR}/videos/player_game.mp4'   # ← غيري هذا

# للتحقق من وجود الفيديو
if not os.path.isfile(PLAYER_VIDEO):
    print('⚠️  الفيديو غير موجود على المسار:', PLAYER_VIDEO)
    print('ارفعيه على Drive أو غيري المسار أعلاه')
else:
    print('✅ الفيديو موجود')

# ── تحميل الفيديو مباشرة من الجهاز (اختياري) ─────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# PLAYER_VIDEO = list(uploaded.keys())[0]

In [ ]:
%cd "{SRC_DIR}"

# ── تشغيل نظام تحليل الأداء ──────────────────────────────────────────────────
!python performance_demo.py \
    --pretrained_path "{PHASE3_CKPT}" \
    --assessment_video "{PLAYER_VIDEO}" \
    --stage_videos "{PLAYER_VIDEO}" \
    --gpu_idx 0 \
    --max_retries 3 \
    --bounce_thresh 0.5 \
    --net_thresh 0.5

## 1️⃣5️⃣ تشغيل مباشر (Demo أصلي)
لمشاهدة تتبع الكرة والـ Segmentation على الفيديو.

In [ ]:
%cd "{SRC_DIR}"

DEMO_OUTPUT = f'{RESULTS_DIR}/demo'
os.makedirs(DEMO_OUTPUT, exist_ok=True)

!python demo.py \
    --working-dir "{PROJECT_DIR}" \
    --pretrained_path "{PHASE3_CKPT}" \
    --video_path "{PLAYER_VIDEO}" \
    --gpu_idx 0 \
    --output_format video \
    --save_demo_output

print('✅ Demo اكتمل. الفيديو محفوظ في:', DEMO_OUTPUT)

## 1️⃣6️⃣ تحميل النتائج
لتحميل الـ checkpoint وفيديو النتائج على جهازك.

In [ ]:
from google.colab import files

# ── تحميل Best Checkpoint ──────────────────────────────────────────────────────
if os.path.isfile(PHASE3_CKPT):
    print('⬇️  تحميل Best Checkpoint...')
    files.download(PHASE3_CKPT)

# ── تحميل فيديو Demo ──────────────────────────────────────────────────────────
demo_video = os.path.join(DEMO_OUTPUT, 'ttnet_phase3', 'result.mp4')
if os.path.isfile(demo_video):
    print('⬇️  تحميل فيديو Demo...')
    files.download(demo_video)
else:
    print('ℹ️  فيديو Demo غير موجود (ربما التدريب لم يكتمل بعد)')

---
## 📋 ملاحظات مهمة

| الموضوع | التفاصيل |
|---------|----------|
| **وقت التدريب** | ~2-4 ساعات لكل مرحلة على T4 GPU |
| **حفظ الجلسة** | كل شيء محفوظ على Drive – لو انقطعت الجلسة تقدري تكملي من آخر checkpoint |
| **استكمال التدريب** | استخدمي `--resume_path` بدل `--pretrained_path` |
| **Colab Pro** | يعطيك وقت جلسة أطول وـ A100 GPU أسرع |
| **الداتا** | ~10 GB كاملة، أو تقدري تشتغلي بـ game_1 فقط للاختبار |

### للاستكمال من checkpoint:
```python
# بدّلي --pretrained_path بـ --resume_path
!python main.py \
    --resume_path "{CHECKPOINTS_DIR}/ttnet_phase1/ttnet_phase1.pth" \
    ...
```